# 🌊 FloodNet CNN Training (Colab Version)

This notebook covers the full pipeline:
1. Downloading a sample flood dataset (or generating synthetic data if downloads fail).
2. Fine-tuning a MobileNetV2 model.
3. Saving the trained model for download.

**Hardware Acceleration:** Go to `Runtime` > `Change runtime type` > Select `T4 GPU`.

In [ ]:
# 1. Setup Environment
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, random_split
from torchvision import transforms, models
from PIL import Image, ImageDraw
import requests
from pathlib import Path
import concurrent.futures
import random
import shutil

print(f"PyTorch Version: {torch.__version__}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using Device: {device}")

In [ ]:
# 2. Download Data (with Synthetic Fallback)

def download_image(url, save_path):
    try:
        response = requests.get(url, timeout=5)
        response.raise_for_status()
        with open(save_path, 'wb') as f:
            f.write(response.content)
        return True
    except Exception:
        return False

def generate_synthetic_image(save_path, is_flood):
    """Generate a dummy image if download fails."""
    color = (50, 50, 200) if is_flood else (50, 200, 50)
    img = Image.new('RGB', (224, 224), color=color)
    draw = ImageDraw.Draw(img)
    # Add random noise
    for _ in range(20):
        x1 = random.randint(0, 224)
        y1 = random.randint(0, 224)
        draw.rectangle([x1, y1, x1+10, y1+10], fill=(255, 255, 255))
    img.save(save_path)
    return True

def setup_dataset():
    base_dir = Path("data/flood_images")
    # Clean up previous run if empty/corrupt
    if base_dir.exists():
        # Start fresh to ensure we get a full set
        shutil.rmtree(base_dir)
        
    (base_dir / "flood").mkdir(parents=True, exist_ok=True)
    (base_dir / "not_flood").mkdir(parents=True, exist_ok=True)
    
    # Sample URLs (Mixed sources)
    FLOOD_URLS = [
        "https://upload.wikimedia.org/wikipedia/commons/thumb/3/30/Flood1_Gneiss.jpg/640px-Flood1_Gneiss.jpg",
        "https://upload.wikimedia.org/wikipedia/commons/thumb/6/6b/Flooded_street_in_New_Orleans_after_Hurricane_Katrina.jpg/640px-Flooded_street_in_New_Orleans_after_Hurricane_Katrina.jpg",
        "https://live.staticflickr.com/5443/9363063541_583858348e_b.jpg"
    ]
    
    NOT_FLOOD_URLS = [
        "https://upload.wikimedia.org/wikipedia/commons/thumb/e/e0/Clouds_over_the_Atlantic_Ocean.jpg/640px-Clouds_over_the_Atlantic_Ocean.jpg",
        "https://upload.wikimedia.org/wikipedia/commons/thumb/9/94/Green_fields.jpg/640px-Green_fields.jpg",
        "https://live.staticflickr.com/3757/12437632615_d840ae0bd6_b.jpg"
    ]
    
    print("Downloading images (or generating synthetic)... ")
    
    # Generate 100 samples total
    target_samples = 50
    
    # Fill Flood Directory
    count = 0
    for i in range(target_samples):
        url = FLOOD_URLS[i % len(FLOOD_URLS)]
        path = base_dir / "flood" / f"flood_{i}.jpg"
        # Try download first, else synthetic
        if not download_image(url, path):
            generate_synthetic_image(path, is_flood=True)
        count += 1
            
    # Fill Not Flood Directory
    for i in range(target_samples):
        url = NOT_FLOOD_URLS[i % len(NOT_FLOOD_URLS)]
        path = base_dir / "not_flood" / f"normal_{i}.jpg"
        if not download_image(url, path):
            generate_synthetic_image(path, is_flood=False)
        count += 1
            
    print(f"Dataset setup complete. Created {count} images in {base_dir}")

setup_dataset()

In [ ]:
# 3. Define Dataset and Model

class FloodDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = Path(root_dir)
        self.transform = transform
        self.samples = []
        
        # Load images
        flood_imgs = list((self.root_dir / "flood").glob("*.jpg"))
        not_flood_imgs = list((self.root_dir / "not_flood").glob("*.jpg"))
        
        for img_path in flood_imgs:
            self.samples.append((str(img_path), 1.0))
        for img_path in not_flood_imgs:
            self.samples.append((str(img_path), 0.0))
            
        print(f"Found {len(flood_imgs)} flood images and {len(not_flood_imgs)} non-flood images")

    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        path, label = self.samples[idx]
        try:
            img = Image.open(path).convert('RGB')
            if self.transform:
                img = self.transform(img)
        except Exception:
            # Fallback for corrupt images
            img = torch.zeros((3, 224, 224))
            
        return img, torch.tensor(label, dtype=torch.float32)

def train_model():
    # Transforms
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    
    dataset = FloodDataset("data/flood_images", transform=transform)
    
    if len(dataset) == 0:
        raise ValueError("No images found! Please run the 'Download Data' cell above.")
        
    train_size = int(0.8 * len(dataset))
    val_size = len(dataset) - train_size
    
    # Handle tiny datasets just in case
    if train_size == 0:
        train_size = len(dataset)
        val_size = 0
    
    train_data, val_data = random_split(dataset, [train_size, val_size])
    
    train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
    val_loader = DataLoader(val_data, batch_size=32) if val_size > 0 else None
    
    # Model (MobileNetV2)
    model = models.mobilenet_v2(weights='IMAGENET1K_V1')
    for param in model.features.parameters():
        param.requires_grad = False
        
    model.classifier = nn.Sequential(
        nn.Dropout(0.3),
        nn.Linear(1280, 1),
        nn.Sigmoid()
    )
    model = model.to(device)
    
    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.classifier.parameters(), lr=0.001)
    
    print("Starting training...")
    for epoch in range(5):
        model.train()
        running_loss = 0.0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs).squeeze()
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
            
        print(f"Epoch {epoch+1}, Loss: {running_loss/len(train_loader):.4f}")
        
    # Configure saving
    model_save_path = "flood_cnn.pth"
    torch.save(model.state_dict(), model_save_path)
    print(f"Model saved to {model_save_path}")
    return model_save_path

model_path = train_model()

In [ ]:
# 4. Download Trained Model
from google.colab import files
if os.path.exists(model_path):
    files.download(model_path)
else:
    print("Model file not found.")